# Kaggle setup (do this once)
1. Kaggle -> Create -> New Notebook -> File -> Import Notebook -> upload this .ipynb
2. Right panel -> **Add Input** -> Competitions -> *State Farm Distracted Driver Detection*
   (accept the competition rules on its page first)
3. Right panel -> Session options -> **Accelerator: GPU T4 x1** (or P100)
4. Run all. Training takes roughly 1.5-3 hours on a T4.
5. Download `output/vgg16_statefarm.pt` from the Output panel and copy it to the
   project's `models/` folder.

To change settings (e.g. the paper's exact random split), edit the `ARGS` cell.

# VGG16 transfer learning on State Farm Distracted Driver Detection

Reproduces the base paper:
  Y. Singh, A. Roy, R. N. Rao, "A Transfer Learning-Based Framework for
  Real-Time Driver Distraction Detection on Edge Devices", IEEE Access, 2026.

Method (paper, Sec. IV):
  * VGG16 pre-trained on ImageNet, include_top=False
  * Head: GlobalAveragePooling -> Dense(512, ReLU) -> Dense(10, softmax)
  * Input 224x224, pixels / 255 then (x - 0.5) / 0.5
  * Augmentation: rotation +-15 deg, shifts +-10 %, horizontal flip
  * Phase 1: backbone frozen, head trained, SGD lr 1e-4, momentum 0.9
  * Phase 2: VGG16 blocks 4 + 5 unfrozen, SGD lr 1e-5, momentum 0.9
  * Batch 8, max 50 epochs, EarlyStopping(patience 8), ReduceLROnPlateau(x0.5),
    checkpoint on best validation accuracy, mixed precision (float16)
  * 80 / 20 stratified split

Two OPTIONAL corrections (flags), recommended for honest evaluation:
  --flip label-swap : a horizontal flip mirrors left/right hands, so c1<->c3 and
                      c2<->c4 must swap labels (the paper flips without swapping)
  --split driver    : hold out whole drivers, so the same person never appears in
                      both train and validation (the paper's random split does)

Run on Kaggle (free GPU): add the "state-farm-distracted-driver-detection"
competition data to the notebook, enable GPU, run all cells.
Output: /kaggle/working/output/vgg16_statefarm.pt (+ metrics, plots)
Copy vgg16_statefarm.pt into the project's models/ folder.

Framework: PyTorch (the paper used TensorFlow/Keras; the model and training
procedure are the same).

In [ ]:
import argparse
import csv
import json
import os
import random
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASSES = [
    "c0", "c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c9",
]
CLASS_NAMES = [
    "Safe driving",
    "Texting - right",
    "Phone call - right",
    "Texting - left",
    "Phone call - left",
    "Operating radio",
    "Drinking",
    "Reaching behind",
    "Hair and makeup",
    "Talking to passenger",
]
# Horizontal mirror swaps the hand used: c1<->c3, c2<->c4.
FLIP_LABEL = {0: 0, 1: 3, 2: 4, 3: 1, 4: 2, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}

INPUT_SIZE = 224
MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]
# torchvision vgg16.features indices: block4 = 17..23, block5 = 24..30
FINE_TUNE_FROM = 17

In [ ]:
# ---------------------------------------------------------------------------
# Model (keep identical to backend/services/detection/behavior_model.py)
# ---------------------------------------------------------------------------

class VGG16Distraction(nn.Module):
    def __init__(self, num_classes: int = 10, pretrained: bool = True):
        super().__init__()
        weights = models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
        self.features = models.vgg16(weights=weights).features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.pool(self.features(x)))


def set_backbone_trainable(model: VGG16Distraction, from_index=None):
    """from_index=None freezes the whole backbone; an index unfreezes from there."""
    for i, layer in enumerate(model.features):
        trainable = from_index is not None and i >= from_index
        for p in layer.parameters():
            p.requires_grad = trainable

In [ ]:
# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def find_data_dir(explicit):
    candidates = [explicit] if explicit else []
    candidates += [
        "/kaggle/input/state-farm-distracted-driver-detection",
        "/kaggle/input/competitions/state-farm-distracted-driver-detection",
        "./state-farm-distracted-driver-detection",
        "./data",
    ]
    for c in candidates:
        if c and (Path(c) / "imgs" / "train").exists():
            return Path(c)
    raise SystemExit(
        "State Farm data not found. Pass --data-dir pointing at the folder that "
        "contains imgs/train/c0..c9 and driver_imgs_list.csv"
    )


def list_samples(data_dir: Path):
    """Returns [(path, label, driver_id)]."""
    drivers = {}
    csv_path = data_dir / "driver_imgs_list.csv"
    if csv_path.exists():
        with open(csv_path, newline="") as f:
            for row in csv.DictReader(f):
                drivers[row["img"]] = row["subject"]

    samples = []
    for label, cls in enumerate(CLASSES):
        for p in sorted((data_dir / "imgs" / "train" / cls).glob("*.jpg")):
            samples.append((p, label, drivers.get(p.name, "unknown")))
    return samples


def split_samples(samples, mode, val_fraction, seed):
    rng = random.Random(seed)

    if mode == "driver":
        ids = sorted({d for _, _, d in samples})
        if len(ids) < 3:
            raise SystemExit("--split driver needs driver_imgs_list.csv")
        rng.shuffle(ids)
        n_val = max(1, round(len(ids) * val_fraction))
        val_ids = set(ids[:n_val])
        train = [s for s in samples if s[2] not in val_ids]
        val = [s for s in samples if s[2] in val_ids]
        return train, val, sorted(val_ids)

    # Stratified random split (paper).
    by_class = defaultdict(list)
    for s in samples:
        by_class[s[1]].append(s)
    train, val = [], []
    for items in by_class.values():
        rng.shuffle(items)
        n_val = round(len(items) * val_fraction)
        val += items[:n_val]
        train += items[n_val:]
    return train, val, []


class StateFarmDataset(Dataset):
    def __init__(self, samples, train: bool, flip_mode: str):
        self.samples = samples
        self.train = train
        self.flip_mode = flip_mode
        resize = transforms.Resize((INPUT_SIZE, INPUT_SIZE))
        normalize = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
        if train:
            self.transform = transforms.Compose(
                [resize, transforms.RandomAffine(degrees=15, translate=(0.1, 0.1))] + normalize
            )
        else:
            self.transform = transforms.Compose([resize] + normalize)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label, _ = self.samples[i]
        image = Image.open(path).convert("RGB")
        if self.train and self.flip_mode != "none" and random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            if self.flip_mode == "label-swap":
                label = FLIP_LABEL[label]
        return self.transform(image), label

In [ ]:
# ---------------------------------------------------------------------------
# Train / evaluate
# ---------------------------------------------------------------------------

def run_epoch(model, loader, device, criterion, optimizer=None, scaler=None, max_batches=None):
    training = optimizer is not None
    model.train(training)
    total, correct, loss_sum = 0, 0, 0.0
    all_pred, all_true = [], []
    use_amp = device.type == "cuda"

    with torch.set_grad_enabled(training):
        for b, (x, y) in enumerate(loader):
            if max_batches is not None and b >= max_batches:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            pred = logits.argmax(1)
            total += y.size(0)
            correct += (pred == y).sum().item()
            loss_sum += loss.item() * y.size(0)
            if not training:
                all_pred.append(pred.cpu())
                all_true.append(y.cpu())

    result = {"loss": loss_sum / max(1, total), "acc": correct / max(1, total)}
    if not training and all_pred:
        result["pred"] = torch.cat(all_pred).numpy()
        result["true"] = torch.cat(all_true).numpy()
    return result


def classification_report(true, pred, n=10):
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    rows = []
    for c in range(n):
        tp = cm[c, c]
        precision = tp / cm[:, c].sum() if cm[:, c].sum() else 0.0
        recall = tp / cm[c, :].sum() if cm[c, :].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({
            "class": CLASSES[c], "name": CLASS_NAMES[c],
            "precision": round(float(precision), 4), "recall": round(float(recall), 4),
            "f1": round(float(f1), 4), "support": int(cm[c, :].sum()),
            "correct": int(tp), "accuracy": round(float(recall) * 100, 2),
        })
    macro = {k: round(float(np.mean([r[k] for r in rows])), 4) for k in ("precision", "recall", "f1")}
    support = np.array([r["support"] for r in rows])
    weighted = {
        k: round(float(np.average([r[k] for r in rows], weights=support)), 4)
        for k in ("precision", "recall", "f1")
    }
    return {"per_class": rows, "macro": macro, "weighted": weighted,
            "accuracy": round(float(np.trace(cm) / cm.sum()), 4), "confusion_matrix": cm.tolist()}


def measure_latency(model, device, runs=30):
    model.eval()
    x = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
    with torch.no_grad():
        for _ in range(3):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(runs):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    return round((time.perf_counter() - t) / runs * 1000, 1)


def save_plots(history, report, out: Path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not available, skipping plots")
        return

    epochs = [h["epoch"] for h in history]
    for key, title in (("loss", "Loss"), ("acc", "Accuracy")):
        plt.figure(figsize=(6, 4))
        plt.plot(epochs, [h[f"train_{key}"] for h in history], label="Train")
        plt.plot(epochs, [h[f"val_{key}"] for h in history], label="Validation")
        p2 = next((h["epoch"] for h in history if h["phase"] == 2), None)
        if p2:
            plt.axvline(p2 - 0.5, color="grey", linestyle="--", linewidth=1, label="Fine-tuning starts")
        plt.xlabel("Epoch"); plt.ylabel(title); plt.title(f"Train and validation {title.lower()}")
        plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(out / f"{key}_curve.png", dpi=150); plt.close()

    cm = np.array(report["confusion_matrix"])
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, cmap="Blues")
    plt.xticks(range(10), CLASSES); plt.yticks(range(10), CLASSES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (validation)")
    for i in range(10):
        for j in range(10):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(fraction=0.046); plt.tight_layout()
    plt.savefig(out / "confusion_matrix.png", dpi=150); plt.close()

In [ ]:
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--out", default="/kaggle/working/output" if Path("/kaggle/working").exists() else "output")
    ap.add_argument("--split", choices=["random", "driver"], default="random",
                    help="random = paper (stratified 80/20); driver = hold out whole drivers")
    ap.add_argument("--flip", choices=["paper", "label-swap", "none"], default="label-swap",
                    help="paper = flip without relabelling; label-swap = swap c1<->c3, c2<->c4 on flip")
    ap.add_argument("--val-fraction", type=float, default=0.2)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--phase1-epochs", type=int, default=5)
    ap.add_argument("--max-epochs", type=int, default=50)
    ap.add_argument("--patience", type=int, default=8)
    ap.add_argument("--lr1", type=float, default=1e-4)
    ap.add_argument("--lr2", type=float, default=1e-5)
    ap.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 1))
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--no-pretrained", action="store_true", help="random init (smoke tests only)")
    ap.add_argument("--max-batches", type=int, default=None, help="limit batches per epoch (smoke tests)")
    args = ap.parse_args(argv)

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu" and args.max_batches is None:
        print("WARNING: no GPU found. Full training on CPU takes days; use a Kaggle/Colab GPU.")

    data_dir = find_data_dir(args.data_dir)
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)

    samples = list_samples(data_dir)
    train_s, val_s, val_drivers = split_samples(samples, args.split, args.val_fraction, args.seed)
    print(f"Device {device} | images {len(samples)} | train {len(train_s)} | val {len(val_s)} "
          f"| split={args.split} flip={args.flip}")
    if val_drivers:
        print("Validation drivers:", ", ".join(val_drivers))

    pin = device.type == "cuda"
    train_loader = DataLoader(StateFarmDataset(train_s, True, args.flip), batch_size=args.batch_size,
                              shuffle=True, num_workers=args.workers, pin_memory=pin, drop_last=False)
    val_loader = DataLoader(StateFarmDataset(val_s, False, "none"), batch_size=max(32, args.batch_size),
                            shuffle=False, num_workers=args.workers, pin_memory=pin)

    model = VGG16Distraction(pretrained=not args.no_pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

    history = []
    best_acc, best_epoch, stale = -1.0, 0, 0
    ckpt = out / "vgg16_statefarm.pt"

    def make_optimizer(lr):
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
        return opt, sched

    # Phase 1: head only
    set_backbone_trainable(model, None)
    optimizer, scheduler = make_optimizer(args.lr1)
    phase = 1

    for epoch in range(1, args.max_epochs + 1):
        if phase == 1 and epoch > args.phase1_epochs:
            # Phase 2: unfreeze blocks 4 and 5
            phase = 2
            set_backbone_trainable(model, FINE_TUNE_FROM)
            optimizer, scheduler = make_optimizer(args.lr2)
            stale = 0
            print(f"--- Phase 2: fine-tuning VGG16 blocks 4-5 at lr {args.lr2}")

        t0 = time.time()
        tr = run_epoch(model, train_loader, device, criterion, optimizer, scaler, args.max_batches)
        va = run_epoch(model, val_loader, device, criterion, max_batches=args.max_batches)
        scheduler.step(va["loss"])

        history.append({
            "epoch": epoch, "phase": phase, "lr": optimizer.param_groups[0]["lr"],
            "train_loss": round(tr["loss"], 4), "train_acc": round(tr["acc"], 4),
            "val_loss": round(va["loss"], 4), "val_acc": round(va["acc"], 4),
            "seconds": round(time.time() - t0, 1),
        })
        print(f"epoch {epoch:02d} p{phase} | train {tr['loss']:.4f}/{tr['acc']:.4f} "
              f"| val {va['loss']:.4f}/{va['acc']:.4f} | {time.time() - t0:.0f}s", flush=True)

        if va["acc"] > best_acc:
            best_acc, best_epoch, stale = va["acc"], epoch, 0
            torch.save({"state_dict": model.state_dict()}, ckpt)
        else:
            stale += 1
            if phase == 2 and stale >= args.patience:
                print(f"Early stopping: no improvement for {args.patience} epochs")
                break

    # Evaluate the best checkpoint
    model.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])
    va = run_epoch(model, val_loader, device, criterion, max_batches=args.max_batches)
    report = classification_report(va["true"], va["pred"])

    latency = {"device": device.type, "ms_per_frame": measure_latency(model, device)}
    if device.type == "cuda":
        latency["cpu_ms_per_frame"] = measure_latency(model.to("cpu"), torch.device("cpu"), runs=5)
        model.to(device)

    meta = {
        "arch": "vgg16-gap-512",
        "classes": CLASSES,
        "class_names": CLASS_NAMES,
        "input_size": INPUT_SIZE,
        "mean": MEAN,
        "std": STD,
        "split": args.split,
        "flip": args.flip,
        "best_epoch": best_epoch,
        "val_accuracy": report["accuracy"],
        "macro_f1": report["macro"]["f1"],
        "latency": latency,
    }
    torch.save({"state_dict": model.state_dict(), "meta": meta}, ckpt)

    with open(out / "metrics.json", "w") as f:
        json.dump({"meta": meta, "report": report, "history": history, "args": vars(args)}, f, indent=2)
    with open(out / "history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        w.writeheader(); w.writerows(history)
    save_plots(history, report, out)

    print("\n=== Validation results (best epoch %d) ===" % best_epoch)
    print(f"Accuracy {report['accuracy'] * 100:.2f}% | macro F1 {report['macro']['f1']:.3f}")
    print(f"{'class':<24}{'prec':>7}{'recall':>8}{'f1':>7}{'n':>6}")
    for r in report["per_class"]:
        print(f"{r['class'] + ' ' + r['name']:<24}{r['precision']:>7.3f}{r['recall']:>8.3f}{r['f1']:>7.3f}{r['support']:>6}")
    print("Latency:", latency)
    print(f"\nSaved {ckpt}  ->  copy it to the project's models/ folder")
    return meta

In [ ]:
ARGS_OUT = "/kaggle/working/output"
# Run 1 (compare with the paper): "random". Run 2 (unseen drivers): "driver".
# Use "--flip", "paper" to reproduce the paper exactly (flip without relabelling).
ARGS = ["--split", "random", "--flip", "label-swap", "--out", ARGS_OUT]
meta = main(ARGS)
meta

In [ ]:
from IPython.display import Image as IPImage, display
for name in ("acc_curve.png", "loss_curve.png", "confusion_matrix.png"):
    display(IPImage(f"{ARGS_OUT}/{name}"))